# Code Indexing and Search

This notebook demonstrates structure-aware code indexing using tree-sitter for
syntax-aware chunking and vector embeddings for semantic search.

In [ ]:
from pathlib import Path

from agentic_patterns.core.connectors.code_repo.connector import CodeRepoConnector
from agentic_patterns.core.doc_ingestion.models import DocumentProvenance
from agentic_patterns.core.rag.chunker_code import ChunkerCode
from agentic_patterns.toolkits.code_index.indexing import index_repo
from agentic_patterns.toolkits.code_index.search import search_and_expand, search_code

## Syntax-aware chunking

ChunkerCode uses tree-sitter to parse source files and extract functions, classes,
and methods as individual chunks.

In [ ]:
sample_code = '''
import os
from pathlib import Path

MAX_RETRIES = 3


def connect(host: str, port: int) -> bool:
    """Establish connection to the server."""
    for attempt in range(MAX_RETRIES):
        try:
            return _try_connect(host, port)
        except OSError:
            continue
    return False


class ConnectionPool:
    def __init__(self, size: int = 10):
        self._pool = []
        self._size = size

    def acquire(self) -> object:
        if self._pool:
            return self._pool.pop()
        return object()

    def release(self, conn: object) -> None:
        if len(self._pool) < self._size:
            self._pool.append(conn)
'''

chunker = ChunkerCode()
provenance = DocumentProvenance(original_file=Path("example.py"), source="example.py")
chunks = chunker.chunk(sample_code, provenance)

for chunk in chunks:
    symbol = chunk.metadata.get("symbol_name", "")
    symbol_type = chunk.metadata.get("symbol_type", "")
    print(f"{chunk.doc_id} [{chunk.level.value}] {symbol_type}:{symbol}")

## Repository scanning

The CodeRepoConnector walks a directory tree, respecting common excludes
like `.git`, `__pycache__`, and `node_modules`.

In [ ]:
connector = CodeRepoConnector()
repo_path = Path(".").resolve().parent.parent.parent  # project root

# Scan only Python files in the core/rag/ directory
files = connector.scan(repo_path / "agentic_patterns" / "core" / "rag", include_patterns=["*.py"])
for f in files:
    print(f.relative_to(repo_path))

## Indexing a repository

Index a subset of the codebase into a Chroma collection for semantic search.

In [ ]:
stats = index_repo(
    repo_path / "agentic_patterns" / "core" / "rag",
    collection_name="code_index_demo",
    include_patterns=["*.py"],
)
print(stats)

## Semantic search

Search the indexed code by meaning rather than keywords.

In [ ]:
hits = search_code("code_index_demo", "split text into chunks", top_k=5)
for hit in hits:
    symbol = hit.metadata.get("symbol_name", "?")
    source = hit.metadata.get("source", "?")
    print(f"[{hit.score:.3f}] {symbol} in {source}")

## Expanded search with cross-references

For the top results, expand context by finding call sites and surrounding code.

In [ ]:
result = search_and_expand(
    "code_index_demo",
    "parse markdown sections",
    repo_path / "agentic_patterns" / "core" / "rag",
    top_k=3,
    expand_top=1,
)
print(result)

## Lexical search

When exact matches matter (variable names, imports), use regex-based search.

In [ ]:
refs = connector.lexical_search(
    repo_path / "agentic_patterns" / "core" / "rag",
    "ChunkerSmart",
    max_results=10,
)
print(refs)